# 06 Major Index Quantum Deep Learning Rolling Windows

This notebook applies the quantum feature pipeline before the deep-learning models.

## Forecast Design
- Target: one-step-ahead index level
- Inputs: 8 technical indicators only
- Dimensionality reduction: none
- Qubit count: `8`, matching the input dimension
- Tune on windows with `2022` test-point dates
- Refit the selected configuration on each evaluation window in `2023-2026`

## Quantum-Enhanced Models
- `MLP`, `RNN`, `LSTM`, `GRU` after quantum feature construction

## Fixed Search Space
- Deep-learning grid: same as notebook `04`
- Quantum settings: `feature_map_reps=[1]`, `anchor_count=[6]`, `shots=[128]`
- Mitigation modes: `ideal`, `aer_noisy`, `mthree_readout`, `zne`


In [ ]:
from __future__ import annotations

import hashlib
import json
import random
import time
from contextlib import contextmanager
from copy import deepcopy
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset

try:
    from qiskit.circuit.library import ZZFeatureMap
    try:
        from qiskit.circuit.library import zz_feature_map
    except Exception:
        zz_feature_map = None
    from qiskit.quantum_info import Statevector
    QISKIT_BASE_AVAILABLE = True
except Exception:
    ZZFeatureMap = None
    zz_feature_map = None
    Statevector = None
    QISKIT_BASE_AVAILABLE = False

try:
    from qiskit_aer import AerSimulator
    from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error
    from qiskit import transpile
    AER_AVAILABLE = True
except Exception:
    AerSimulator = None
    NoiseModel = None
    ReadoutError = None
    depolarizing_error = None
    transpile = None
    AER_AVAILABLE = False

try:
    import mthree
    MTHREE_AVAILABLE = True
except Exception:
    mthree = None
    MTHREE_AVAILABLE = False


def resolve_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Run this notebook from the project root or its notebooks directory.")


PROJECT_ROOT = resolve_project_root()
PIPELINE_BUNDLE_PATH = PROJECT_ROOT / "outputs" / "01_major_index_sliding_window_pipeline" / "major_index_sliding_window_bundle.joblib"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "06_major_index_quantum_deep_learning_rolling"
LOG_DIR = OUTPUT_ROOT / "rolling_logs"
PREDICTIONS_DIR = OUTPUT_ROOT / "predictions"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIG_DIR = OUTPUT_ROOT / "figures"
for directory in [OUTPUT_ROOT, LOG_DIR, PREDICTIONS_DIR, TABLE_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TARGET_COL = "target_close"
FEATURE_COLS = ["rsi14", "macd_hist", "bb_z20", "roc10", "vol20", "sma10_gap", "atr14_pct", "williams_r14"]
NUM_QUBITS = len(FEATURE_COLS)
RANDOM_SEED = 42
ZNE_FOLD_SCALES = [1, 3, 5]
SHOTS = 128
STATEVECTOR_BATCH_SIZE = 16
ALL_LOGS_PATH = OUTPUT_ROOT / "all_rolling_logs.csv"
SCORES_SUMMARY_PATH = TABLE_DIR / "scores_summary.csv"
BEST_CONFIGS_PATH = TABLE_DIR / "best_configs.csv"
MODEL_SUMMARY_PATH = TABLE_DIR / "model_summary.csv"
GRID_PROFILE_PATH = TABLE_DIR / "grid_profile.csv"
TIMING_SUMMARY_PATH = TABLE_DIR / "timing_summary.csv"

REQUESTED_MITIGATION_MODES = ["ideal", "aer_noisy", "mthree_readout", "zne"]
ACTIVE_MITIGATION_MODES = ["ideal"]
if AER_AVAILABLE:
    ACTIVE_MITIGATION_MODES.append("aer_noisy")
    ACTIVE_MITIGATION_MODES.append("zne")
if AER_AVAILABLE and MTHREE_AVAILABLE:
    ACTIVE_MITIGATION_MODES.append("mthree_readout")

MODEL_ORDER = ["quantum_MLP", "quantum_RNN", "quantum_LSTM", "quantum_GRU"]
QUANTUM_GRID_SPEC = {
    "feature_map_reps": [1],
    "anchor_count": [6],
    "shots": [SHOTS],
    "mitigation_mode": ACTIVE_MITIGATION_MODES,
}
DEEP_GRID_SPEC = {
    "hidden_dim": [32, 64, 128],
    "num_layers": [1, 2, 3],
    "learning_rate": [1e-3, 3e-4, 1e-4],
    "activation": ["relu", "tanh", "silu"],
    "weight_decay": [0.0, 1e-4, 1e-3],
    "dropout": [0.0],
    "batch_size": [16],
    "epoch": [32],
}
def format_grid_spec(spec: dict) -> str:
    return " | ".join(f"{key}={list(values)}" for key, values in spec.items())


QUANTUM_GRID = list(ParameterGrid(QUANTUM_GRID_SPEC))
DEEP_GRID = list(ParameterGrid(DEEP_GRID_SPEC))
COMBINED_CONFIGS = [{"quantum_params": q, "deep_params": d} for q in QUANTUM_GRID for d in DEEP_GRID]
GRID_PROFILE = pd.DataFrame(
    [
        {
            "model_name": model_name,
            "n_quantum_configs": int(len(QUANTUM_GRID)),
            "n_deep_configs": int(len(DEEP_GRID)),
            "n_combined_configs": int(len(COMBINED_CONFIGS)),
            "quantum_spec": format_grid_spec(QUANTUM_GRID_SPEC),
            "active_mitigation_modes": ", ".join(ACTIVE_MITIGATION_MODES),
        }
        for model_name in MODEL_ORDER
    ]
).sort_values("model_name").reset_index(drop=True)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

TIMING_ROWS: list[dict] = []
FORCE_FRESH_RUN = False
FLUSH_EVERY_N_ROWS = 1
MODEL_SELECTION_SCOPE = "tuning_windows_only"
EXPORT_PREDICTION_SCOPE = "all_logged"  # all_logged | pure_test_only
PLOT_PREDICTION_SCOPE = "all_logged"  # all_logged | pure_test_only
PLOT_SHADE_TUNING_PERIOD = True
HP_TUNING_LABEL = "Used for HP tuning"
PURE_TEST_LABEL = "Pure test period"
PURE_TEST_START_DATE = "2023-01-01"


In [ ]:
@contextmanager
def timed_step(step_name: str):
    start = time.perf_counter()
    yield
    elapsed_seconds = time.perf_counter() - start
    TIMING_ROWS.append({"step": step_name, "elapsed_seconds": elapsed_seconds})
    print(f"[TIMER] {step_name}: {elapsed_seconds:.2f}s")


def set_seeds(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_case_bundle(path: Path = PIPELINE_BUNDLE_PATH) -> dict:
    bundle = joblib.load(path)
    return bundle["case_bundle"]


LOG_DATE_COLS = ["train_end_date", "test_reference_date", "test_target_date"]
WINDOW_KEY_COLS = ["case_key", "model_name", "config_index", "run_stage", "window_id"]


def reset_output_targets() -> None:
    for path in [ALL_LOGS_PATH, SCORES_SUMMARY_PATH, BEST_CONFIGS_PATH, MODEL_SUMMARY_PATH, TIMING_SUMMARY_PATH]:
        if path.exists():
            path.unlink()
    for directory in [LOG_DIR, PREDICTIONS_DIR]:
        for csv_path in directory.glob("*.csv"):
            csv_path.unlink()
    for png_path in FIG_DIR.glob("*.png"):
        png_path.unlink()


def append_frame(path: Path, frame: pd.DataFrame) -> None:
    if frame is None or frame.empty:
        return
    frame.to_csv(path, mode="a", header=not path.exists(), index=False)


def load_progress_frame(path: Path = ALL_LOGS_PATH) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path)
    for col in LOG_DATE_COLS:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")
    if "is_tuning_window" in frame.columns:
        frame["is_tuning_window"] = frame["is_tuning_window"].astype(str).str.lower().eq("true")
    if all(col in frame.columns for col in WINDOW_KEY_COLS):
        frame = frame.drop_duplicates(subset=WINDOW_KEY_COLS, keep="last").reset_index(drop=True)
    return frame


def window_result_key(case_key: str, model_name: str, config_index: int, run_stage: str, window_id: int) -> tuple[str, str, int, str, int]:
    return (str(case_key), str(model_name), int(config_index), str(run_stage), int(window_id))


def completed_window_keys(frame: pd.DataFrame, run_stage: str | None = None) -> set[tuple[str, str, int, str, int]]:
    if frame.empty or "status" not in frame.columns:
        return set()
    completed = frame.loc[frame["status"].eq("completed")].copy()
    if run_stage is not None and "run_stage" in completed.columns:
        completed = completed.loc[completed["run_stage"].eq(run_stage)].copy()
    if completed.empty:
        return set()
    return {
        window_result_key(
            row["case_key"],
            row["model_name"],
            int(row["config_index"]),
            row["run_stage"],
            int(row["window_id"]),
        )
        for _, row in completed.iterrows()
    }


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    yt = np.asarray(y_true, dtype=float)[mask]
    yp = np.asarray(y_pred, dtype=float)[mask]
    if len(yt) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "n_points": 0}
    return {
        "MAE": float(mean_absolute_error(yt, yp)),
        "RMSE": float(np.sqrt(mean_squared_error(yt, yp))),
        "R2": float(r2_score(yt, yp)) if len(yt) > 1 else np.nan,
        "n_points": int(len(yt)),
    }


def add_error_columns(frame: pd.DataFrame) -> pd.DataFrame:
    if frame is None or len(frame) == 0:
        return frame.copy() if isinstance(frame, pd.DataFrame) else pd.DataFrame()
    enriched = frame.copy()
    enriched["error_points"] = enriched["predicted_close"] - enriched["actual_close"]
    enriched["error_pct"] = np.where(
        np.isfinite(enriched["actual_close"]) & (np.abs(enriched["actual_close"]) > 1e-12),
        100.0 * enriched["error_points"] / enriched["actual_close"],
        np.nan,
    )
    enriched["abs_error_pct"] = np.abs(enriched["error_pct"])
    return enriched


def summarize_quantum_params(config: dict) -> str:
    return " ".join(
        [
            f"reps={int(config['feature_map_reps'])}",
            f"anchors={int(config['anchor_count'])}",
            f"shots={int(config['shots'])}",
            f"mit={config['mitigation_mode']}",
        ]
    )


def summarize_deep_params(config: dict) -> str:
    return " ".join(
        [
            f"hid={int(config['hidden_dim'])}",
            f"layers={int(config['num_layers'])}",
            f"drop={float(config['dropout']):.2f}",
            f"lr={float(config['learning_rate']):.0e}",
            f"bs={int(config['batch_size'])}",
            f"ep={int(config['epoch'])}",
            f"act={config['activation']}",
            f"wd={float(config['weight_decay']):.0e}",
        ]
    )


def summarize_config(quantum_params: dict, deep_params: dict) -> str:
    return f"{summarize_quantum_params(quantum_params)} | {summarize_deep_params(deep_params)}"


def config_key(model_name: str, quantum_params: dict, deep_params: dict) -> str:
    payload = json.dumps({"model": model_name, **quantum_params, **deep_params}, sort_keys=True)
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]


def fit_quantum_preprocessor(X_train: np.ndarray) -> dict:
    scaler = StandardScaler().fit(X_train)
    X_scaled = scaler.transform(X_train)
    mins = X_scaled.min(axis=0)
    maxs = X_scaled.max(axis=0)
    spans = np.where((maxs - mins) < 1e-8, 1.0, (maxs - mins))
    return {"scaler": scaler, "mins": mins, "spans": spans}


def transform_to_angles(bundle: dict, X: np.ndarray) -> np.ndarray:
    X_scaled = bundle["scaler"].transform(X)
    angles = np.pi * (X_scaled - bundle["mins"]) / bundle["spans"]
    angles = np.nan_to_num(angles, nan=0.0, posinf=np.pi, neginf=0.0)
    return np.clip(angles, 0.0, np.pi).astype(np.float32)


def build_feature_map(feature_dimension: int, reps: int):
    if zz_feature_map is not None:
        return zz_feature_map(feature_dimension=feature_dimension, reps=int(reps), entanglement="linear")
    return ZZFeatureMap(feature_dimension=feature_dimension, reps=int(reps), entanglement="linear")


def choose_anchor_indices(n_train: int, anchor_count: int) -> np.ndarray:
    if anchor_count >= n_train:
        return np.arange(n_train, dtype=np.int32)
    return np.unique(np.linspace(0, n_train - 1, num=int(anchor_count), dtype=np.int32))


def compute_statevector_anchor_features(angle_matrix: np.ndarray, anchor_angle_matrix: np.ndarray, reps: int) -> np.ndarray:
    feature_map = build_feature_map(angle_matrix.shape[1], reps=reps)
    anchor_statevectors = []
    for row in anchor_angle_matrix:
        circuit = feature_map.assign_parameters(row.tolist(), inplace=False)
        anchor_statevectors.append(Statevector.from_instruction(circuit).data)
    anchor_statevectors = np.asarray(anchor_statevectors, dtype=np.complex128)
    rows = []
    for start in range(0, len(angle_matrix), STATEVECTOR_BATCH_SIZE):
        batch = angle_matrix[start : start + STATEVECTOR_BATCH_SIZE]
        batch_states = []
        for row in batch:
            circuit = feature_map.assign_parameters(row.tolist(), inplace=False)
            batch_states.append(Statevector.from_instruction(circuit).data)
        batch_states = np.asarray(batch_states, dtype=np.complex128)
        overlaps = batch_states @ anchor_statevectors.conj().T
        rows.append((np.abs(overlaps) ** 2).astype(np.float32))
    return np.vstack(rows)


def build_generic_noise_model(num_qubits: int, scale_factor: float = 1.0) -> NoiseModel:
    model = NoiseModel()
    readout_p = min(max(0.03 * float(scale_factor), 1e-6), 0.49)
    readout = ReadoutError([[1.0 - readout_p, readout_p], [readout_p, 1.0 - readout_p]])
    for qubit in range(int(num_qubits)):
        model.add_readout_error(readout, [qubit])
    error_1q = depolarizing_error(min(0.001 * float(scale_factor), 0.2), 1)
    error_2q = depolarizing_error(min(0.01 * float(scale_factor), 0.2), 2)
    for gate in ["rz", "sx", "x", "p", "h", "u", "u1", "u2", "u3"]:
        try:
            model.add_all_qubit_quantum_error(error_1q, [gate])
        except Exception:
            pass
    for gate in ["cx", "ecr", "cz"]:
        try:
            model.add_all_qubit_quantum_error(error_2q, [gate])
        except Exception:
            pass
    return model


def add_measurements(circuit):
    measured = circuit.copy()
    measured.measure_all()
    return measured


def counts_to_probability_vector(counts: dict, num_qubits: int) -> np.ndarray:
    vec = np.zeros(2 ** int(num_qubits), dtype=np.float64)
    total = max(float(sum(counts.values())), 1.0)
    for bitstring, count in counts.items():
        idx = int(str(bitstring).replace(" ", ""), 2)
        vec[idx] = float(count) / total
    denom = max(vec.sum(), 1e-12)
    return vec / denom


def distribution_to_probability_vector(distribution, num_qubits: int) -> np.ndarray:
    vec = np.zeros(2 ** int(num_qubits), dtype=np.float64)
    for state, value in dict(distribution).items():
        if isinstance(state, (int, np.integer)):
            idx = int(state)
        else:
            idx = int(str(state).replace(" ", ""), 2)
        vec[idx] = float(value)
    denom = max(vec.sum(), 1e-12)
    return vec / denom


def bhattacharyya_similarity(prob_matrix: np.ndarray, anchor_prob_matrix: np.ndarray) -> np.ndarray:
    similarities = np.sqrt(prob_matrix[:, None, :] * anchor_prob_matrix[None, :, :]).sum(axis=2)
    return np.clip(similarities ** 2, 0.0, 1.0).astype(np.float32)


def simulate_prob_features(
    angle_matrix: np.ndarray,
    anchor_angle_matrix: np.ndarray,
    reps: int,
    mitigation_mode: str = "aer_noisy",
    scale_factor: float = 1.0,
    shots: int = SHOTS,
) -> np.ndarray:
    feature_map = build_feature_map(angle_matrix.shape[1], reps=reps)
    simulator = AerSimulator(noise_model=build_generic_noise_model(angle_matrix.shape[1], scale_factor=scale_factor))
    measured_qubits = list(range(int(angle_matrix.shape[1])))
    mitigation = None
    if mitigation_mode == "mthree_readout":
        mitigation = mthree.M3Mitigation(simulator)
        mitigation.cals_from_system(measured_qubits)
    rows = []
    anchor_rows = []
    for matrix, out_rows in ((angle_matrix, rows), (anchor_angle_matrix, anchor_rows)):
        for row in matrix:
            circuit = feature_map.assign_parameters(row.tolist(), inplace=False)
            measured = add_measurements(circuit)
            compiled = transpile(measured, simulator, optimization_level=0)
            result = simulator.run(compiled, shots=int(shots), seed_simulator=RANDOM_SEED + int(scale_factor * 10)).result()
            counts = result.get_counts(0)
            if mitigation_mode == "mthree_readout":
                quasi = mitigation.apply_correction(counts, measured_qubits)
                try:
                    prob_distribution = quasi.nearest_probability_distribution()
                except Exception:
                    try:
                        prob_distribution = quasi.probabilities()
                    except Exception:
                        prob_distribution = dict(quasi)
                out_rows.append(distribution_to_probability_vector(prob_distribution, angle_matrix.shape[1]))
            else:
                out_rows.append(counts_to_probability_vector(counts, angle_matrix.shape[1]))
    return bhattacharyya_similarity(np.asarray(rows), np.asarray(anchor_rows))


def zne_extrapolate(scales: list[int], values: np.ndarray) -> np.ndarray:
    scales_arr = np.asarray(scales, dtype=np.float64)
    out = np.zeros(values.shape[1:], dtype=np.float64)
    for i in range(values.shape[1]):
        for j in range(values.shape[2]):
            coeffs = np.polyfit(scales_arr, values[:, i, j], 1)
            out[i, j] = np.polyval(coeffs, 0.0)
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def prepare_quantum_features(X_train: np.ndarray, X_other: np.ndarray, quantum_params: dict) -> dict:
    if not QISKIT_BASE_AVAILABLE:
        raise ImportError("qiskit is required for the quantum workflow.")
    bundle = fit_quantum_preprocessor(X_train)
    train_angles = transform_to_angles(bundle, X_train)
    other_angles = transform_to_angles(bundle, X_other)
    anchor_indices = choose_anchor_indices(len(train_angles), int(quantum_params["anchor_count"]))
    anchor_angles = train_angles[anchor_indices]
    mitigation_mode = str(quantum_params["mitigation_mode"])
    reps = int(quantum_params["feature_map_reps"])
    shots = int(quantum_params["shots"])

    if mitigation_mode == "ideal":
        train_anchor = compute_statevector_anchor_features(train_angles, anchor_angles, reps=reps)
        other_anchor = compute_statevector_anchor_features(other_angles, anchor_angles, reps=reps)
    elif mitigation_mode == "aer_noisy":
        if not AER_AVAILABLE:
            raise ImportError("qiskit_aer is required for aer_noisy.")
        train_anchor = simulate_prob_features(train_angles, anchor_angles, reps=reps, mitigation_mode="aer_noisy", scale_factor=1.0, shots=shots)
        other_anchor = simulate_prob_features(other_angles, anchor_angles, reps=reps, mitigation_mode="aer_noisy", scale_factor=1.0, shots=shots)
    elif mitigation_mode == "zne":
        if not AER_AVAILABLE:
            raise ImportError("qiskit_aer is required for zne.")
        scaled_train = []
        scaled_other = []
        for scale in ZNE_FOLD_SCALES:
            scaled_train.append(simulate_prob_features(train_angles, anchor_angles, reps=reps, mitigation_mode="aer_noisy", scale_factor=float(scale), shots=shots))
            scaled_other.append(simulate_prob_features(other_angles, anchor_angles, reps=reps, mitigation_mode="aer_noisy", scale_factor=float(scale), shots=shots))
        train_anchor = zne_extrapolate(ZNE_FOLD_SCALES, np.stack(scaled_train, axis=0))
        other_anchor = zne_extrapolate(ZNE_FOLD_SCALES, np.stack(scaled_other, axis=0))
    elif mitigation_mode == "mthree_readout":
        if not AER_AVAILABLE or not MTHREE_AVAILABLE:
            raise ImportError("qiskit_aer and mthree are required for mthree_readout.")
        train_anchor = simulate_prob_features(train_angles, anchor_angles, reps=reps, mitigation_mode="mthree_readout", scale_factor=1.0, shots=shots)
        other_anchor = simulate_prob_features(other_angles, anchor_angles, reps=reps, mitigation_mode="mthree_readout", scale_factor=1.0, shots=shots)
    else:
        raise ValueError(mitigation_mode)

    return {
        "bundle": bundle,
        "anchor_indices": anchor_indices,
        "train_anchor_features": train_anchor.astype(np.float32),
        "other_anchor_features": other_anchor.astype(np.float32),
    }


def prepare_model_inputs(anchor_features: np.ndarray, model_name: str) -> np.ndarray:
    if model_name == "quantum_MLP":
        return anchor_features.astype(np.float32)
    return anchor_features.astype(np.float32)[:, :, None]


def _act(name: str):
    if name == "relu":
        return nn.ReLU()
    if name == "tanh":
        return nn.Tanh()
    if name == "silu":
        return nn.SiLU()
    raise ValueError(name)


class QuantumHead(nn.Module):
    def __init__(self, model_name: str, input_dim: int, params: dict):
        super().__init__()
        hidden_dim = int(params["hidden_dim"])
        num_layers = int(params["num_layers"])
        dropout = float(params["dropout"])
        act = params["activation"]

        if model_name == "quantum_MLP":
            blocks = []
            current_dim = input_dim
            for _ in range(num_layers):
                blocks.extend([nn.Linear(current_dim, hidden_dim), _act(act), nn.Dropout(dropout)])
                current_dim = hidden_dim
            self.backbone = nn.Sequential(*blocks)
            self.head = nn.Linear(current_dim, 1)
            self.kind = "MLP"
        else:
            recurrent_cls = {"quantum_RNN": nn.RNN, "quantum_LSTM": nn.LSTM, "quantum_GRU": nn.GRU}[model_name]
            self.rnn = recurrent_cls(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, dropout=dropout if num_layers > 1 else 0.0, batch_first=True)
            self.drop = nn.Dropout(dropout)
            self.head = nn.Linear(hidden_dim, 1)
            self.kind = "RNN"

    def forward(self, x):
        if self.kind == "MLP":
            return self.head(self.backbone(x)).squeeze(-1)
        out, _ = self.rnn(x)
        return self.head(self.drop(out[:, -1, :])).squeeze(-1)


def fit_predict_quantum_window(model_name: str, deep_params: dict, X_train_ready: np.ndarray, y_train: np.ndarray, X_test_ready: np.ndarray) -> float:
    set_seeds(RANDOM_SEED)
    y_scaler = StandardScaler().fit(y_train.reshape(-1, 1))
    y_train_scaled = y_scaler.transform(y_train.reshape(-1, 1)).ravel().astype(np.float32)
    model = QuantumHead(model_name=model_name, input_dim=X_train_ready.shape[-1], params=deep_params).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(deep_params["learning_rate"]), weight_decay=float(deep_params["weight_decay"]))
    loss_fn = nn.MSELoss()
    dataset = TensorDataset(torch.tensor(X_train_ready, dtype=torch.float32), torch.tensor(y_train_scaled, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=min(int(deep_params["batch_size"]), len(dataset)), shuffle=False)
    model.train()
    for _ in range(int(deep_params["epoch"])):
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
    model.eval()
    with torch.no_grad():
        xt = torch.tensor(X_test_ready, dtype=torch.float32, device=DEVICE)
        pred = model(xt).detach().cpu().numpy().reshape(-1, 1)
    return float(y_scaler.inverse_transform(pred).ravel()[0])


def run_manifest_for_config(
    *,
    case_key: str,
    index_key: str,
    window_size: int,
    frame: pd.DataFrame,
    manifest: pd.DataFrame,
    model_name: str,
    config_index: int,
    quantum_params: dict,
    deep_params: dict,
    run_stage: str,
    progress_bar=None,
    completed_keys: set[tuple[str, str, int, str, int]] | None = None,
    flush_path: Path | None = None,
    flush_every: int = FLUSH_EVERY_N_ROWS,
) -> list[dict]:
    rows = []
    buffer: list[dict] = []
    cfg_key = config_key(model_name, quantum_params, deep_params)
    cfg_summary = summarize_config(quantum_params, deep_params)
    quantum_json = json.dumps(quantum_params, sort_keys=True)
    deep_json = json.dumps(deep_params, sort_keys=True)
    for _, meta in manifest.iterrows():
        unit_key = window_result_key(case_key, model_name, int(config_index), run_stage, int(meta["window_id"]))
        if completed_keys is not None and unit_key in completed_keys:
            continue

        train_df = frame.iloc[int(meta["start"]) : int(meta["start"]) + int(meta["train_rows"])].copy()
        test_df = frame.iloc[int(meta["end_exclusive"]) - int(meta["test_rows"]) : int(meta["end_exclusive"])].copy()
        X_train = train_df[FEATURE_COLS].to_numpy(dtype=np.float32)
        X_test = test_df[FEATURE_COLS].to_numpy(dtype=np.float32)
        y_train = train_df[TARGET_COL].to_numpy(dtype=np.float32)
        y_true = float(test_df[TARGET_COL].iloc[-1])
        reference_close = float(test_df["reference_close"].iloc[-1])
        target_close = float(test_df["target_close"].iloc[-1])
        fit_start = time.perf_counter()
        status = "completed"
        error_message = ""
        y_pred = np.nan
        try:
            quantum_bundle = prepare_quantum_features(X_train=X_train, X_other=X_test, quantum_params=quantum_params)
            X_train_ready = prepare_model_inputs(quantum_bundle["train_anchor_features"], model_name=model_name)
            X_test_ready = prepare_model_inputs(quantum_bundle["other_anchor_features"], model_name=model_name)
            y_pred = fit_predict_quantum_window(model_name=model_name, deep_params=deep_params, X_train_ready=X_train_ready, y_train=y_train, X_test_ready=X_test_ready)
        except Exception as exc:
            status = "failed"
            error_message = f"{type(exc).__name__}: {exc}"
        fit_seconds = time.perf_counter() - fit_start
        row = {
            "case_key": case_key,
            "index_key": index_key,
            "window_size": window_size,
            "model_name": model_name,
            "config_index": int(config_index),
            "config_key": cfg_key,
            "config_summary": cfg_summary,
            "quantum_params_json": quantum_json,
            "deep_params_json": deep_json,
            "run_stage": run_stage,
            "window_id": int(meta["window_id"]),
            "train_end_date": pd.Timestamp(meta["train_end_date"]),
            "test_reference_date": pd.Timestamp(meta["test_reference_date"]),
            "test_target_date": pd.Timestamp(meta["test_target_date"]),
            "is_tuning_window": bool(meta["is_tuning_window"]),
            "reference_close": reference_close,
            "target_close": target_close,
            "y_true": y_true,
            "y_pred": float(y_pred) if np.isfinite(y_pred) else np.nan,
            "actual_close": target_close if np.isfinite(target_close) else np.nan,
            "predicted_close": float(y_pred) if np.isfinite(y_pred) else np.nan,
            "fit_seconds": float(fit_seconds),
            "status": status,
            "error_message": error_message,
        }
        rows.append(row)
        buffer.append(row)
        if status == "completed" and completed_keys is not None:
            completed_keys.add(unit_key)
        if flush_path is not None and len(buffer) >= int(flush_every):
            append_frame(flush_path, finalize_logs(buffer))
            buffer = []
        if progress_bar is not None:
            progress_bar.update(1)
    if flush_path is not None and buffer:
        append_frame(flush_path, finalize_logs(buffer))
    return rows


def finalize_logs(rows: list[dict]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    frame = frame.sort_values(["case_key", "model_name", "config_index", "window_id"]).reset_index(drop=True)
    return add_error_columns(frame)


def split_completed_logs(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    return frame.loc[frame["status"].eq("completed")].copy()



def build_scores_summary(logs: pd.DataFrame) -> pd.DataFrame:
    completed = split_completed_logs(logs)
    rows = []
    group_cols = ["case_key", "index_key", "window_size", "model_name", "config_index", "config_key", "config_summary"]
    for keys, group in completed.groupby(group_cols, dropna=False):
        tune = add_error_columns(group.loc[group["is_tuning_window"]].copy())
        eval_frame = add_error_columns(group.loc[~group["is_tuning_window"]].copy())
        m_tune = regression_metrics(tune["y_true"].to_numpy(dtype=float), tune["y_pred"].to_numpy(dtype=float))
        m_eval = regression_metrics(eval_frame["y_true"].to_numpy(dtype=float), eval_frame["y_pred"].to_numpy(dtype=float))
        tune_mean_close = float(tune["actual_close"].mean()) if len(tune) else np.nan
        eval_mean_close = float(eval_frame["actual_close"].mean()) if len(eval_frame) else np.nan
        rows.append(
            {
                "case_key": keys[0],
                "index_key": keys[1],
                "window_size": int(keys[2]),
                "model": keys[3],
                "config_index": int(keys[4]),
                "config_key": keys[5],
                "config_summary": keys[6],
                "quantum_params_json": group["quantum_params_json"].iloc[0],
                "deep_params_json": group["deep_params_json"].iloc[0],
                "tune_MAE": m_tune["MAE"],
                "tune_RMSE": m_tune["RMSE"],
                "tune_R2": m_tune["R2"],
                "tune_n": m_tune["n_points"],
                "tune_close_MAE": m_tune["MAE"],
                "tune_close_RMSE": m_tune["RMSE"],
                "tune_close_R2": m_tune["R2"],
                "tune_close_n": m_tune["n_points"],
                "tune_close_MAE_pct": 100.0 * m_tune["MAE"] / tune_mean_close if np.isfinite(tune_mean_close) and abs(tune_mean_close) > 1e-12 else np.nan,
                "tune_close_RMSE_pct": 100.0 * m_tune["RMSE"] / tune_mean_close if np.isfinite(tune_mean_close) and abs(tune_mean_close) > 1e-12 else np.nan,
                "tune_abs_error_pct_mean": float(tune["abs_error_pct"].mean()) if len(tune) else np.nan,
                "tune_error_pct_bias": float(tune["error_pct"].mean()) if len(tune) else np.nan,
                "tune_windows": int(len(tune)),
                "eval_MAE": m_eval["MAE"],
                "eval_RMSE": m_eval["RMSE"],
                "eval_R2": m_eval["R2"],
                "eval_n": m_eval["n_points"],
                "eval_close_MAE": m_eval["MAE"],
                "eval_close_RMSE": m_eval["RMSE"],
                "eval_close_R2": m_eval["R2"],
                "eval_close_n": m_eval["n_points"],
                "eval_close_MAE_pct": 100.0 * m_eval["MAE"] / eval_mean_close if np.isfinite(eval_mean_close) and abs(eval_mean_close) > 1e-12 else np.nan,
                "eval_close_RMSE_pct": 100.0 * m_eval["RMSE"] / eval_mean_close if np.isfinite(eval_mean_close) and abs(eval_mean_close) > 1e-12 else np.nan,
                "eval_abs_error_pct_mean": float(eval_frame["abs_error_pct"].mean()) if len(eval_frame) else np.nan,
                "eval_error_pct_bias": float(eval_frame["error_pct"].mean()) if len(eval_frame) else np.nan,
                "eval_windows": int(len(eval_frame)),
                "mean_fit_seconds": float(group["fit_seconds"].mean()),
                "metric_scope": "selection=one_step_index_level; presentation=index_level_and_relative_error",
                "target_definition": "one_step_index_level",
            }
        )
    summary = pd.DataFrame(rows)
    if summary.empty:
        return summary
    return summary.sort_values(["case_key", "model", "config_index"]).reset_index(drop=True)



def select_best_configs(scores: pd.DataFrame) -> pd.DataFrame:
    picks = []
    for (case_key, model_name), group in scores.groupby(["case_key", "model"], as_index=False):
        group = group.dropna(subset=["tune_close_RMSE"]).copy()
        if group.empty:
            continue
        pick = group.sort_values(
            ["tune_close_RMSE", "tune_abs_error_pct_mean", "tune_close_MAE", "config_index"],
            ascending=[True, True, True, True],
            kind="stable",
        ).iloc[0]
        picks.append(pick.to_dict())
    picked = pd.DataFrame(picks)
    if picked.empty:
        return picked
    return picked.sort_values(["case_key", "model"]).reset_index(drop=True)


In [ ]:
GRID_PROFILE.to_csv(GRID_PROFILE_PATH, index=False)
display(GRID_PROFILE)


with timed_step("tuning_quantum_dl_runs"):
    case_bundle = load_case_bundle()
    if FORCE_FRESH_RUN:
        reset_output_targets()

    existing_logs = load_progress_frame()
    completed_tuning = completed_window_keys(existing_logs, run_stage="tuning")
    tuning_total_full = 0
    for case in case_bundle.values():
        tuning_windows = int(case["window_manifest"]["is_tuning_window"].sum())
        tuning_total_full += tuning_windows * len(MODEL_ORDER) * len(COMBINED_CONFIGS)
    tuning_total_pending = max(0, tuning_total_full - len(completed_tuning))

    with tqdm(total=tuning_total_pending, desc="Quantum DL tuning windows") as progress_bar:
        for case_key, case in case_bundle.items():
            frame = case["df"].copy().reset_index(drop=True)
            tuning_manifest = case["window_manifest"].loc[case["window_manifest"]["is_tuning_window"]].copy().reset_index(drop=True)

            for model_name in MODEL_ORDER:
                progress_bar.set_postfix_str(f"{case_key} | {model_name}", refresh=False)
                for config_index, combo in enumerate(COMBINED_CONFIGS, start=1):
                    run_manifest_for_config(
                        case_key=case_key,
                        index_key=case["index_key"],
                        window_size=int(case["window_size"]),
                        frame=frame,
                        manifest=tuning_manifest,
                        model_name=model_name,
                        config_index=int(config_index),
                        quantum_params=combo["quantum_params"],
                        deep_params=combo["deep_params"],
                        run_stage="tuning",
                        progress_bar=progress_bar,
                        completed_keys=completed_tuning,
                        flush_path=ALL_LOGS_PATH,
                    )

    tuning_logs = load_progress_frame()
    tuning_logs = tuning_logs.loc[tuning_logs["run_stage"].eq("tuning")].copy()


with timed_step("select_best_quantum_dl_configs"):
    tuning_scores = build_scores_summary(tuning_logs)
    selected_configs = select_best_configs(tuning_scores)
    tuning_scores.to_csv(SCORES_SUMMARY_PATH, index=False)
    selected_configs.to_csv(BEST_CONFIGS_PATH, index=False)


with timed_step("evaluation_quantum_dl_runs"):
    existing_logs = load_progress_frame()
    completed_eval = completed_window_keys(existing_logs, run_stage="evaluation")
    evaluation_total_pending = 0
    for _, selected_row in selected_configs.iterrows():
        case_key = selected_row["case_key"]
        case = case_bundle[case_key]
        eval_manifest = case["window_manifest"].loc[~case["window_manifest"]["is_tuning_window"]].copy().reset_index(drop=True)
        for window_id in eval_manifest["window_id"].tolist():
            unit_key = window_result_key(case_key, selected_row["model"], int(selected_row["config_index"]), "evaluation", int(window_id))
            if unit_key not in completed_eval:
                evaluation_total_pending += 1

    with tqdm(total=evaluation_total_pending, desc="Quantum DL evaluation windows") as progress_bar:
        for _, selected_row in selected_configs.iterrows():
            case_key = selected_row["case_key"]
            case = case_bundle[case_key]
            frame = case["df"].copy().reset_index(drop=True)
            eval_manifest = case["window_manifest"].loc[~case["window_manifest"]["is_tuning_window"]].copy().reset_index(drop=True)
            progress_bar.set_postfix_str(f"{case_key} | {selected_row['model']}", refresh=False)
            run_manifest_for_config(
                case_key=case_key,
                index_key=case["index_key"],
                window_size=int(case["window_size"]),
                frame=frame,
                manifest=eval_manifest,
                model_name=selected_row["model"],
                config_index=int(selected_row["config_index"]),
                quantum_params=json.loads(selected_row["quantum_params_json"]),
                deep_params=json.loads(selected_row["deep_params_json"]),
                run_stage="evaluation",
                progress_bar=progress_bar,
                completed_keys=completed_eval,
                flush_path=ALL_LOGS_PATH,
            )

    rolling_logs = load_progress_frame()
    if not rolling_logs.empty:
        rolling_logs = rolling_logs.sort_values(["case_key", "model_name", "config_index", "window_id"]).reset_index(drop=True)
        rolling_logs.to_csv(ALL_LOGS_PATH, index=False)
        for (case_key, model_name), group in rolling_logs.groupby(["case_key", "model_name"], as_index=False):
            group.to_csv(LOG_DIR / f"{case_key}_{model_name}.csv", index=False)

print("Selected configs from tuning stage")
display(selected_configs)
print("Rolling-log sample")
display(rolling_logs.head(20))


In [ ]:

with timed_step("score_and_export_summaries"):
    scores_summary = build_scores_summary(rolling_logs)
    best_configs = select_best_configs(scores_summary)

    prediction_rows = []
    for _, best_row in best_configs.iterrows():
        subset = rolling_logs.loc[
            rolling_logs["case_key"].eq(best_row["case_key"])
            & rolling_logs["model_name"].eq(best_row["model"])
            & rolling_logs["config_key"].eq(best_row["config_key"])
            & rolling_logs["status"].eq("completed")
        ].copy()
        subset["analysis_partition"] = np.where(subset["is_tuning_window"], "hp_tuning", "pure_test")
        subset["analysis_scope"] = np.where(subset["is_tuning_window"], "used_for_hyperparameter_selection", "held_out_pure_test")
        subset = add_error_columns(subset)
        if EXPORT_PREDICTION_SCOPE == "pure_test_only":
            subset = subset.loc[subset["analysis_partition"].eq("pure_test")].copy()
        elif EXPORT_PREDICTION_SCOPE != "all_logged":
            raise ValueError(EXPORT_PREDICTION_SCOPE)

        prediction_path = PREDICTIONS_DIR / f"{best_row['case_key']}_{best_row['model']}_{best_row['config_key']}.csv"
        subset.to_csv(prediction_path, index=False)
        row = dict(best_row)
        row["prediction_path"] = str(prediction_path)
        row["prediction_export_scope"] = EXPORT_PREDICTION_SCOPE
        prediction_rows.append(row)

    model_summary = pd.DataFrame(prediction_rows)
    if not model_summary.empty:
        model_summary = model_summary.sort_values(
            ["case_key", "eval_close_RMSE", "eval_abs_error_pct_mean", "eval_close_MAE", "mean_fit_seconds"],
            ascending=[True, True, True, True, True],
        ).reset_index(drop=True)
    scores_summary.to_csv(SCORES_SUMMARY_PATH, index=False)
    best_configs.to_csv(BEST_CONFIGS_PATH, index=False)
    model_summary.to_csv(MODEL_SUMMARY_PATH, index=False)

timing_summary = pd.DataFrame(TIMING_ROWS).sort_values("elapsed_seconds", ascending=False).reset_index(drop=True)
timing_summary.to_csv(TIMING_SUMMARY_PATH, index=False)

print("Best configs")
display(best_configs)
print("Model summary")
display(model_summary)
print("Timing summary")
display(timing_summary)


In [ ]:

with timed_step("build_summary_figures"):
    summary_key = "case_key"
    summary_figure_path = FIG_DIR / "01_model_overview.png"
    line_figure_path = FIG_DIR / "02_best_case_level_paths.png"
    error_figure_path = FIG_DIR / "03_best_case_error_pct_paths.png"
    if model_summary.empty:
        print("No completed runs available for summary figures.")
    else:
        avg_metrics = (
            model_summary.groupby("model", as_index=False)[["eval_close_MAE", "eval_close_RMSE", "eval_abs_error_pct_mean", "mean_fit_seconds"]]
            .mean(numeric_only=True)
            .sort_values(["eval_close_RMSE", "eval_abs_error_pct_mean"])
            .reset_index(drop=True)
        )
        heatmap_frame = model_summary.pivot(index=summary_key, columns="model", values="eval_close_RMSE").sort_index()

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        axes[0].bar(avg_metrics["model"], avg_metrics["eval_close_MAE"], color="#2f6b82")
        axes[0].set_title("Average Evaluation MAE")
        axes[0].set_ylabel("Index-Level MAE")
        axes[0].tick_params(axis="x", rotation=30)

        axes[1].bar(avg_metrics["model"], avg_metrics["eval_abs_error_pct_mean"], color="#4f5866")
        axes[1].set_title("Average Absolute Daily Error")
        axes[1].set_ylabel("% of Actual Index Level")
        axes[1].tick_params(axis="x", rotation=30)

        im = axes[2].imshow(heatmap_frame.to_numpy(dtype=float), aspect="auto", cmap="Blues")
        axes[2].set_title("Evaluation RMSE by Case")
        axes[2].set_xticks(range(len(heatmap_frame.columns)))
        axes[2].set_xticklabels(list(heatmap_frame.columns), rotation=30, ha="right")
        axes[2].set_yticks(range(len(heatmap_frame.index)))
        axes[2].set_yticklabels(list(heatmap_frame.index))
        fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

        fig.tight_layout()
        fig.savefig(summary_figure_path, dpi=200, bbox_inches="tight")
        plt.show()
        display(avg_metrics.round(6))
        display(heatmap_frame.round(6))

        best_overall = (
            model_summary.sort_values(
                ["case_key", "eval_close_RMSE", "eval_abs_error_pct_mean", "eval_close_MAE", "mean_fit_seconds"],
                ascending=[True, True, True, True, True],
            )
            .groupby("case_key", as_index=False)
            .head(1)
            .reset_index(drop=True)
        )

        fig, axes = plt.subplots(4, 2, figsize=(18, 14), sharex=False)
        axes = np.asarray(axes).reshape(-1)
        for ax, (_, best_row) in zip(axes, best_overall.iterrows()):
            pred_frame = pd.read_csv(best_row["prediction_path"])
            pred_frame["test_target_date"] = pd.to_datetime(pred_frame["test_target_date"], errors="coerce")
            pred_frame = pred_frame.sort_values("test_target_date").reset_index(drop=True)
            pred_frame = add_error_columns(pred_frame)
            if PLOT_PREDICTION_SCOPE == "pure_test_only":
                pred_frame = pred_frame.loc[pred_frame["analysis_partition"].eq("pure_test")].copy()
            elif PLOT_PREDICTION_SCOPE != "all_logged":
                raise ValueError(PLOT_PREDICTION_SCOPE)

            ax.plot(pred_frame["test_target_date"], pred_frame["actual_close"], label="Actual", color="#1f4e79", linewidth=1.6)
            ax.plot(pred_frame["test_target_date"], pred_frame["predicted_close"], label="Predicted", color="#c00000", linewidth=1.3, alpha=0.85)

            if PLOT_SHADE_TUNING_PERIOD and PLOT_PREDICTION_SCOPE == "all_logged":
                tune_mask = pred_frame["analysis_partition"].eq("hp_tuning")
                pure_mask = pred_frame["analysis_partition"].eq("pure_test")
                if int(tune_mask.sum()) > 0:
                    tune_end = pred_frame.loc[tune_mask, "test_target_date"].max()
                    tune_start = pred_frame.loc[tune_mask, "test_target_date"].min()
                    ax.axvspan(tune_start, tune_end, color="#d9dde3", alpha=0.25, label=HP_TUNING_LABEL)
                if int(pure_mask.sum()) > 0:
                    pure_start = pred_frame.loc[pure_mask, "test_target_date"].min()
                    ax.axvline(pure_start, color="#808080", linestyle="--", linewidth=1.0, label=PURE_TEST_LABEL)

            ax.set_title(f"{best_row['case_key']} | {best_row['model']}")
            ax.grid(alpha=0.25)
        for ax in axes[len(best_overall):]:
            ax.axis("off")
        handles, labels = axes[0].get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        fig.legend(unique.values(), unique.keys(), loc="upper center", ncol=min(4, len(unique)), frameon=False)
        fig.tight_layout(rect=[0, 0, 1, 0.97])
        fig.savefig(line_figure_path, dpi=200, bbox_inches="tight")
        plt.show()

        fig, axes = plt.subplots(4, 2, figsize=(18, 14), sharex=False)
        axes = np.asarray(axes).reshape(-1)
        for ax, (_, best_row) in zip(axes, best_overall.iterrows()):
            pred_frame = pd.read_csv(best_row["prediction_path"])
            pred_frame["test_target_date"] = pd.to_datetime(pred_frame["test_target_date"], errors="coerce")
            pred_frame = pred_frame.sort_values("test_target_date").reset_index(drop=True)
            pred_frame = add_error_columns(pred_frame)
            if PLOT_PREDICTION_SCOPE == "pure_test_only":
                pred_frame = pred_frame.loc[pred_frame["analysis_partition"].eq("pure_test")].copy()
            elif PLOT_PREDICTION_SCOPE != "all_logged":
                raise ValueError(PLOT_PREDICTION_SCOPE)

            ax.plot(pred_frame["test_target_date"], pred_frame["error_pct"], color="#c00000", linewidth=1.2, alpha=0.9)
            ax.axhline(0.0, color="#111111", linestyle="--", linewidth=0.9)

            if PLOT_SHADE_TUNING_PERIOD and PLOT_PREDICTION_SCOPE == "all_logged":
                tune_mask = pred_frame["analysis_partition"].eq("hp_tuning")
                pure_mask = pred_frame["analysis_partition"].eq("pure_test")
                if int(tune_mask.sum()) > 0:
                    tune_end = pred_frame.loc[tune_mask, "test_target_date"].max()
                    tune_start = pred_frame.loc[tune_mask, "test_target_date"].min()
                    ax.axvspan(tune_start, tune_end, color="#d9dde3", alpha=0.25, label=HP_TUNING_LABEL)
                if int(pure_mask.sum()) > 0:
                    pure_start = pred_frame.loc[pure_mask, "test_target_date"].min()
                    ax.axvline(pure_start, color="#808080", linestyle="--", linewidth=1.0, label=PURE_TEST_LABEL)

            ax.set_title(f"{best_row['case_key']} | {best_row['model']}")
            ax.set_ylabel("Prediction Error (%)")
            ax.grid(alpha=0.25)
        for ax in axes[len(best_overall):]:
            ax.axis("off")
        handles, labels = axes[0].get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        fig.legend(unique.values(), unique.keys(), loc="upper center", ncol=min(4, len(unique)), frameon=False)
        fig.tight_layout(rect=[0, 0, 1, 0.97])
        fig.savefig(error_figure_path, dpi=200, bbox_inches="tight")
        plt.show()
        display(best_overall[["case_key", "model", "eval_close_RMSE", "eval_close_MAE", "eval_abs_error_pct_mean"]].round(6))
